# ds004920 on Neurodesk: download one subject, visualize T1w and BOLD data with Niivue, and create FSL 3-column timing files

This example notebook is written in a **Neurodesk EDU** style for students who are beginning to work with open fMRI data in a BIDS dataset. The notebook does three things:

1. downloads **one subject** from **ds004920** for anatomical and functional visualization,
2. visualizes a **T1w image** and **one BOLD run** with **Niivue** inside Jupyter, and
3. converts **all available BIDS `_events.tsv` files** in the dataset into **FSL-style 3-column timing files**.

The goal is to bridge the logic in introductory labs on MRI viewing and first-level modeling with a clean, reproducible notebook workflow that students can adapt for their own projects.

**Author:** David V. Smith  
**Date:** 2026-03-25  
**License:** MIT License

> Note: if this notebook uses tools provided through Neurodesk containers, those tools retain their original licenses. See the Neurodesk citation guidance for details.

### Citation and Resources

#### Tools included in this workflow
- **DataLad** for lightweight dataset installation and selective file retrieval: https://www.datalad.org/
- **FSL**: Jenkinson, M., Beckmann, C. F., Behrens, T. E. J., Woolrich, M. W., & Smith, S. M. (2012). *FSL*. *NeuroImage, 62*(2), 782–790. https://doi.org/10.1016/j.neuroimage.2011.09.015
- **Niivue / IPyNiiVue** for browser-based neuroimaging visualization in Jupyter: https://niivue.github.io/ipyniivue/
- **NiBabel** for working with NIfTI images in Python: https://nipy.org/nibabel/

#### Publications
- Gorgolewski, K. J., et al. (2016). *The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments*. *Scientific Data, 3*, 160044. https://doi.org/10.1038/sdata.2016.44

#### Educational resources
- This example is conceptually aligned with course materials on viewing MRI data and creating 3-column timing files for first-level modeling in FSL.
- Neurodesk example notebook template: https://neurodesk.org/edu/examples/template.html

#### Dataset
- **OpenNeuro ds004920, version 1.1.1**: *An fMRI dataset of social and nonsocial reward processing in young adults*. https://openneuro.org/datasets/ds004920/versions/1.1.1
- Smith, D. V., Wyngaarden, J., Sharp, C. J., Sazhin, D., Zaff, O., Fareri, D., & Jarcho, J. (2024). *An fMRI dataset of social and nonsocial reward processing in young adults*. *Data in Brief, 53*, 110197. https://doi.org/10.1016/j.dib.2024.110197

## Table of content
[1. Load software tools and import Python libraries](#1.-Load-software-tools-and-import-Python-libraries)  
[2. Parameters and helper functions](#2.-Parameters-and-helper-functions)  
[3. Install the dataset and fetch the files we need](#3.-Install-the-dataset-and-fetch-the-files-we-need)  
[4. Summarize the dataset files available locally](#4.-Summarize-the-dataset-files-available-locally)  
[5. Visualize the T1w image and one BOLD run with Niivue](#5.-Visualize-the-T1w-image-and-one-BOLD-run-with-Niivue)  
[6. Convert all BIDS events files to FSL 3-column timing files](#6.-Convert-all-BIDS-events-files-to-FSL-3-column-timing-files)  
[7. Inspect the generated timing files](#7.-Inspect-the-generated-timing-files)  
[8. Dependencies in Jupyter/Python](#8.-Dependencies-in-Jupyter/Python)

## 1. Load software tools and import Python libraries

In [ ]:
%%capture
!pip install -q ipyniivue nibabel pandas numpy matplotlib watermark

In [ ]:
import module
await module.load('fsl/6.0.7.16')
await module.list()

In [ ]:
from ipyniivue import NiiVue
from IPython.display import Markdown, display
from pathlib import Path
import json
import os
import re
import subprocess

import nibabel as nib
import numpy as np
import pandas as pd

os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"

## 2. Parameters and helper functions

The notebook downloads a **single subject's imaging data** for visualization but fetches **all task event files** so we can generate 3-column timing files across the whole dataset.

We keep all derived files in `derivatives/` so the raw BIDS data stay untouched.

In [ ]:
DATASET_URL = "https://github.com/OpenNeuroDatasets/ds004920.git"
DATASET_TAG = "1.1.1"
DATASET_NAME = "ds004920"

SELECTED_SUBJECT = "sub-1001"
SELECTED_BOLD_BASENAME = f"{SELECTED_SUBJECT}_task-doors_run-1_bold.nii.gz"

HOME = Path.home()
DATASET_ROOT = HOME / DATASET_NAME
DERIVATIVES_ROOT = DATASET_ROOT / "derivatives"
PREVIEW_ROOT = DERIVATIVES_ROOT / "notebook_preview"
THREECOL_ROOT = DERIVATIVES_ROOT / "fsl_3col"

PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)
THREECOL_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def parse_bids_entities(path):
    name = Path(path).name.replace("_events.tsv", "")
    entities = {}
    for token in name.split("_"):
        if "-" in token:
            key, value = token.split("-", 1)
            entities[key] = value
    return entities

def sanitize_label(label):
    label = str(label).strip()
    label = re.sub(r"\s+", "_", label)
    label = re.sub(r"[^A-Za-z0-9._-]", "_", label)
    label = re.sub(r"_+", "_", label).strip("._-")
    return label or "condition"

def subject_from_path(path):
    path = Path(path)
    for part in path.parts:
        if re.fullmatch(r"sub-[A-Za-z0-9]+", part):
            return part
    match = re.search(r"(sub-[A-Za-z0-9]+)", path.name)
    return match.group(1) if match else "unknown-sub"

## 3. Install the dataset and fetch the files we need

In [ ]:
if not DATASET_ROOT.exists():
    run(f"datalad install {DATASET_URL}", cwd=HOME)

run("git fetch --tags", cwd=DATASET_ROOT)
run(f"git checkout {DATASET_TAG}", cwd=DATASET_ROOT)

run(f"datalad get {SELECTED_SUBJECT}/anat/{SELECTED_SUBJECT}_T1w.nii.gz", cwd=DATASET_ROOT)
run(f"datalad get {SELECTED_SUBJECT}/func/{SELECTED_BOLD_BASENAME}", cwd=DATASET_ROOT)

# Fetch every events file in the dataset so we can generate 3-column timing files
run("datalad get sub-*/func/*_events.tsv", cwd=DATASET_ROOT)

## 4. Summarize the dataset files available locally

The ds004920 README describes a young-adult sample that completed **four fMRI tasks**: `ugdg`, `sharedreward`, `socialdoors`, and `mid`. This cell checks what is currently available after the file retrieval step above.

In [ ]:
t1_path = DATASET_ROOT / SELECTED_SUBJECT / "anat" / f"{SELECTED_SUBJECT}_T1w.nii.gz"
bold_path = DATASET_ROOT / SELECTED_SUBJECT / "func" / SELECTED_BOLD_BASENAME
bold_json_path = DATASET_ROOT / SELECTED_SUBJECT / "func" / SELECTED_BOLD_BASENAME.replace(".nii.gz", ".json")

events_files = sorted(DATASET_ROOT.glob("sub-*/func/*_events.tsv"))
event_entities = [parse_bids_entities(p) for p in events_files]
events_manifest = pd.DataFrame(event_entities).fillna("")

task_counts = (
    events_manifest.groupby("task")
    .size()
    .reset_index(name="n_event_files")
    .sort_values(["n_event_files", "task"], ascending=[False, True])
)

summary_text = f"""
**Selected subject for visualization:** `{SELECTED_SUBJECT}`  
**Selected BOLD run:** `{SELECTED_BOLD_BASENAME}`  
**Number of local `_events.tsv` files:** **{len(events_files)}**
"""
display(Markdown(summary_text))

display(task_counts)
events_manifest.head(10)

## 5. Visualize the T1w image and one BOLD run with Niivue

We first inspect the anatomical image and one functional run for `sub-1001`. The BOLD file is 4D, so we also create a representative **middle volume** for a lighter-weight overlay display.

In [ ]:
t1_img = nib.load(str(t1_path))
bold_img = nib.load(str(bold_path))

with open(bold_json_path) as f:
    bold_meta = json.load(f)

middle_vol_idx = bold_img.shape[3] // 2
middle_bold_data = np.asarray(bold_img.dataobj[..., middle_vol_idx], dtype=np.float32)
middle_bold_img = nib.Nifti1Image(middle_bold_data, bold_img.affine, bold_img.header)

middle_bold_path = PREVIEW_ROOT / SELECTED_BOLD_BASENAME.replace(".nii.gz", f"_middle-vol-{middle_vol_idx:03d}.nii.gz")
nib.save(middle_bold_img, str(middle_bold_path))

t1_zooms = t1_img.header.get_zooms()[:3]
bold_zooms = bold_img.header.get_zooms()[:4]

meta_text = f"""
### Quick metadata summary

- **T1w shape:** `{t1_img.shape}`
- **T1w voxel size (mm):** `{tuple(round(x, 3) for x in t1_zooms)}`
- **BOLD shape:** `{bold_img.shape}`
- **BOLD voxel size + TR:** `{tuple(round(x, 3) for x in bold_zooms)}`
- **BOLD RepetitionTime (s):** `{bold_meta.get("RepetitionTime", "n/a")}`
- **BOLD TaskName:** `{bold_meta.get("TaskName", "n/a")}`
- **Representative BOLD volume for overlays:** volume index `{middle_vol_idx}`
"""
display(Markdown(meta_text))

In [ ]:
# T1w anatomy
nv = NiiVue()
nv.load_volumes([
    {"path": str(t1_path), "name": f"{SELECTED_SUBJECT} T1w", "colormap": "gray"}
])
nv

In [ ]:
# Full 4D BOLD run
# In many Jupyter frontends, Niivue will expose a time/volume control for 4D images.
nv = NiiVue()
nv.load_volumes([
    {"path": str(bold_path), "name": SELECTED_BOLD_BASENAME, "colormap": "gray"}
])
nv

In [ ]:
# Optional overlay: T1w anatomy with a representative middle BOLD volume in red
nv = NiiVue()
nv.load_volumes([
    {"path": str(t1_path), "name": "T1w", "colormap": "gray"},
    {"path": str(middle_bold_path), "name": "BOLD middle volume", "colormap": "red", "opacity": 0.6},
])
nv

## 6. Convert all BIDS events files to FSL 3-column timing files

FSL timing files contain three columns:

1. **onset** in seconds  
2. **duration** in seconds  
3. **weight** (usually `1`)

In this dataset, the events files include a `trial_type` column. We create **one 3-column text file per unique `trial_type`** and write the outputs into:

`derivatives/fsl_3col/sub-XXXX/func/`

This approach preserves the raw BIDS dataset while producing FEAT-ready timing files.

In [ ]:
def convert_events_to_three_col(events_path, out_root):
    events_path = Path(events_path)
    df = pd.read_csv(events_path, sep="\t")

    required = {"onset", "duration"}
    if not required.issubset(df.columns):
        raise ValueError(f"{events_path} is missing one of the required columns: {required}")

    if "trial_type" in df.columns:
        grouping = df["trial_type"].fillna("n/a").astype(str)
    else:
        grouping = pd.Series(["all"] * len(df), index=df.index)

    subject = subject_from_path(events_path)
    entities = parse_bids_entities(events_path)
    task = entities.get("task", "unknown-task")
    run = entities.get("run", "")
    entity_root = events_path.name.replace("_events.tsv", "")

    out_dir = Path(out_root) / subject / "func"
    out_dir.mkdir(parents=True, exist_ok=True)

    records = []
    for raw_label in pd.unique(grouping):
        subset = df.loc[grouping == raw_label, ["onset", "duration"]].copy()
        subset["weight"] = 1

        safe_label = sanitize_label(raw_label)
        out_file = out_dir / f"{entity_root}_trial-{safe_label}.txt"
        subset.to_csv(out_file, sep="\t", header=False, index=False, float_format="%.6f")

        records.append(
            {
                "subject": subject,
                "task": task,
                "run": run,
                "trial_type": raw_label,
                "n_events": len(subset),
                "events_tsv": str(events_path.relative_to(DATASET_ROOT)),
                "three_col_file": str(out_file.relative_to(DATASET_ROOT)),
            }
        )

    return records

In [ ]:
threecol_records = []
for events_path in events_files:
    threecol_records.extend(convert_events_to_three_col(events_path, THREECOL_ROOT))

threecol_manifest = pd.DataFrame(threecol_records).sort_values(
    ["subject", "task", "run", "trial_type"]
).reset_index(drop=True)

manifest_path = THREECOL_ROOT / "manifest.tsv"
threecol_manifest.to_csv(manifest_path, sep="\t", index=False)

manifest_text = f"""
Created **{len(threecol_manifest)}** 3-column files from **{len(events_files)}** events files.  
A manifest was written to: `{manifest_path.relative_to(DATASET_ROOT)}`
"""
display(Markdown(manifest_text))

threecol_manifest.head(20)

## 7. Inspect the generated timing files

In [ ]:
summary_by_task = (
    threecol_manifest.groupby(["task", "trial_type"])
    .agg(
        n_threecol_files=("three_col_file", "count"),
        total_events=("n_events", "sum"),
    )
    .reset_index()
    .sort_values(["task", "trial_type"])
)

display(summary_by_task)

example_row = threecol_manifest.query(
    "subject == @SELECTED_SUBJECT and task == 'doors' and run == '1' and trial_type == 'win'"
).iloc[0]

example_threecol = DATASET_ROOT / example_row["three_col_file"]
display(Markdown(f"### Example output file\n`{example_row['three_col_file']}`"))

pd.read_csv(
    example_threecol,
    sep="\t",
    header=None,
    names=["onset", "duration", "weight"]
).head(10)

### Notes for students

- These text files are ready to use as **custom 3-column EVs** in FSL FEAT.
- Because the outputs are written under `derivatives/`, the original BIDS dataset remains unchanged.
- For first-level modeling, you would usually decide **which trial types to keep separate** and which to combine based on your hypothesis. The notebook intentionally writes one file per observed condition so that the design decisions stay transparent.

## 8. Dependencies in Jupyter/Python

In [ ]:
%load_ext watermark
%watermark -iv

print("\nCommand line tools:")
run("datalad --version")
run("git --version")